# AI-Powered Engineering Study Assistant (Capstone)

Full RAG + Agent + Memory + Tool

## Capstone Plan

Domain: Engineering Study Assistant

User: Engineering students

Success: Accurate RAG answers, memory support, tool usage

In [2]:
!pip install chromadb sentence-transformers langchain-groq langgraph python-dotenv -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [3]:
import os
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq


In [4]:
DOCUMENTS = [
{'id':'1','text':'Ohm law V=IR'},
{'id':'2','text':'Kirchhoff laws circuits'},
{'id':'3','text':'SHM periodic motion'},
{'id':'4','text':'Thermodynamics heat'},
{'id':'5','text':'Wave motion energy'},
{'id':'6','text':'Capacitance charge'},
{'id':'7','text':'Inductance magnetic'},
{'id':'8','text':'Semiconductors electronics'},
{'id':'9','text':'Logic gates'},
{'id':'10','text':'Control systems'}
]

In [6]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.Client()
collection = client.create_collection('capstone_kb')
texts=[d['text'] for d in DOCUMENTS]
ids=[d['id'] for d in DOCUMENTS]
emb=embedder.encode(texts).tolist()
collection.add(documents=texts, embeddings=emb, ids=ids)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
import os
os.environ["GROQ_API_KEY"] = "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"

In [8]:
groq_key = os.getenv('GROQ_API_KEY','')
llm = ChatGroq(model='llama-3.1-8b-instant', groq_api_key=groq_key)


In [9]:
def retrieve(query):
    q_emb = embedder.encode([query]).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=2)
    return res['documents'][0]


In [10]:
def answer(query):
    docs = retrieve(query)
    context = '\n'.join(docs)
    prompt = f'Answer using context:\n{context}\nQ:{query}'
    return llm.invoke(prompt).content


In [11]:
def tool_time():
    import datetime
    return str(datetime.datetime.now())


In [12]:
memory = {}
def chat(user, query):
    if 'time' in query.lower():
        return tool_time()
    memory[user] = query
    return answer(query)


In [13]:
print(chat('nikhil','What is Ohm law?'))
print(chat('nikhil','what is time now?'))


Ohm's Law is a fundamental principle in electricity that states the relationship between voltage (V), current (I), and resistance (R) in an electrical circuit. It is expressed as:

V = IR

Where:

- V is the voltage in volts (V)
- I is the current in amperes (A)
- R is the resistance in ohms (Ω)

Ohm's Law describes how voltage, current, and resistance are interconnected, and it's a crucial concept in understanding and designing electrical circuits.
2026-04-20 14:01:30.935301


## Reflection

This system integrates Retrieval-Augmented Generation (RAG) with a Large Language Model (LLM) to provide accurate and context-based answers. Instead of relying on general knowledge, the model retrieves relevant documents from a structured knowledge base using embeddings stored in ChromaDB.

The use of memory allows the system to retain user interactions, enabling more personalized and context-aware conversations. Additionally, tool integration, such as the date and time function, enhances the system’s capability to handle real-time queries beyond static knowledge retrieval.

Overall, the combination of RAG, memory, and tool usage makes the system more reliable, reduces hallucination, and improves the quality of responses. This architecture demonstrates how modern AI systems can be designed to solve domain-specific problems effectively.